# 02 — Pré-processamento dos Dados

**Mineração de Dados Aplicada à Predição de Inadimplência em Clientes de Cartão de Crédito**  
UPE — Escola Politécnica de Pernambuco

Implementa a fase de *Data Preparation* do CRISP-DM descrita na **Seção 3.4** do artigo.

Pipeline: **limpeza → transformação → redução → split → escalonamento**.
A ordem importa: o split antecede o escalonamento e qualquer balanceamento, para impedir vazamento de dados.

In [1]:
# ---------------------------------------------------------------------------
# Setup — funciona local, no Google Colab e no Kaggle.
# Se estiver rodando fora do repositório, o bloco abaixo faz o clone.
# ---------------------------------------------------------------------------
import os, sys, subprocess

REPO_URL = "https://github.com/SEU-USUARIO/inadimplencia-cartao-credito.git"
REPO_DIR = "inadimplencia-cartao-credito"

def localizar_raiz():
    # Sobe até encontrar a pasta data/raw (execução dentro do repositório)
    d = os.path.abspath(os.getcwd())
    for _ in range(4):
        if os.path.isdir(os.path.join(d, "data", "raw")):
            return d
        d = os.path.dirname(d)
    return None

raiz = localizar_raiz()
if raiz is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    raiz = os.path.abspath(REPO_DIR)
os.chdir(raiz)
sys.path.insert(0, os.path.join(raiz, "src"))
print("Raiz do projeto:", raiz)

RAW_PATH = os.path.join("data", "raw", "default_of_credit_card_clients.xls")
assert os.path.exists(RAW_PATH), (
    "Base bruta não encontrada. Baixe 'default of credit card clients.xls' do "
    "repositório da UCI (https://doi.org/10.24432/C55S3H) e salve em data/raw/."
)


Raiz do projeto: /home/claude/proj/repo


In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
RANDOM_STATE = 42
TARGET = "default payment next month"


In [3]:
log = {}
df = pd.read_excel(RAW_PATH, header=1)
log["linhas_brutas"], log["colunas_brutas"] = df.shape
log["valores_ausentes"] = int(df.isna().sum().sum())
print(f"[1] Base bruta: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"[1] Valores ausentes: {log['valores_ausentes']}")

[1] Base bruta: 30000 linhas x 25 colunas
[1] Valores ausentes: 0


## 1. Limpeza

**1.1 Remoção do `ID`.** Contador sequencial de 1 a 30.000, sem relação com o comportamento do cliente; mantido, seria lido como variável numérica ordenada.

In [4]:
df = df.drop(columns=["ID"])
print("Colunas:", df.shape[1])

Colunas: 24


**1.2 Duplicatas.** Perfis idênticos nas 24 colunas restantes. Se a mesma observação caísse em treino e em teste, a métrica de validação ficaria otimista.

In [5]:
n_before = len(df)
df = df.drop_duplicates()
log["duplicatas_removidas"] = n_before - len(df)
log["linhas_apos_limpeza"] = len(df)
print(f"[2] Duplicatas removidas: {log['duplicatas_removidas']} -> {len(df)} registros")

[2] Duplicatas removidas: 35 -> 29965 registros


**1.3 Padronização de nomenclatura.** `PAY_0` → `PAY_1`, alinhando o índice mensal com `BILL_AMT1..6` e `PAY_AMT1..6`.

In [6]:
df = df.rename(columns={"PAY_0": "PAY_1"})

**1.4 Categorias não documentadas.** `EDUCATION` ∈ {0, 5, 6} e `MARRIAGE` = 0 não constam do dicionário oficial. São consolidados nas categorias residuais (4 e 3), e não excluídos: o problema está na documentação, não no registro.

In [7]:
log["education_nao_documentado"] = int(df["EDUCATION"].isin([0, 5, 6]).sum())
log["marriage_nao_documentado"] = int((df["MARRIAGE"] == 0).sum())
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})
print(f"[4] EDUCATION recodificados: {log['education_nao_documentado']} | "
      f"MARRIAGE recodificados: {log['marriage_nao_documentado']}")

[4] EDUCATION recodificados: 345 | MARRIAGE recodificados: 54


**1.5 Outliers.** Identificados pelo critério IQR e **mantidos**: são clientes de limite alto e idade mais avançada, subgrupo real da população. A escala é tratada no bloco 5 e os modelos de árvore são robustos a esses valores.

In [8]:
for col in ["LIMIT_BAL", "AGE"]:
    q1, q3 = df[col].quantile([0.25, 0.75]); iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = int(((df[col] < lo) | (df[col] > hi)).sum())
    log[f"outliers_iqr_{col}"] = n_out
    print(f"[5] {col}: {n_out} outliers mantidos (limites {lo:,.1f} a {hi:,.1f})")

[5] LIMIT_BAL: 167 outliers mantidos (limites -235,000.0 a 525,000.0)
[5] AGE: 272 outliers mantidos (limites 8.5 a 60.5)


## 2. Transformação (engenharia de atributos)

**2.1 `AVG_UTIL_RATIO`** — utilização média do limite. O valor absoluto da fatura tem correlação ~0 com o alvo; a razão fatura/limite torna clientes de limites diferentes comparáveis. Valores negativos são preservados: indicam saldo credor.

In [9]:
pay_cols = [f"PAY_{i}" for i in range(1, 7)]
bill_cols = [f"BILL_AMT{i}" for i in range(1, 7)]
payamt_cols = [f"PAY_AMT{i}" for i in range(1, 7)]

df["AVG_UTIL_RATIO"] = df[bill_cols].mean(axis=1) / df["LIMIT_BAL"]
df["AVG_UTIL_RATIO"].describe().round(3)

count    29965.000
mean         0.373
std          0.352
min         -0.233
25%          0.030
50%          0.286
75%          0.688
max          5.364
Name: AVG_UTIL_RATIO, dtype: float64

**2.2 `AVG_PAY_RATIO` e `SEM_FATURA_POSITIVA`** — taxa média de amortização.

A razão `PAY_AMTi / BILL_AMTi` só faz sentido quando existe dívida a amortizar. Meses com fatura zero (sem consumo) ou negativa (saldo credor) são **excluídos da média**, e não convertidos em valor absoluto — tomar o módulo de uma fatura negativa inverteria o significado financeiro do indicador.

Para os clientes sem nenhuma fatura positiva no período, o indicador é indefinido. Atribuir 0 sem mais os tornaria indistinguíveis de quem devia e nada pagou — o oposto em termos de risco. Daí a flag explícita.

In [10]:
bill_pos = df[bill_cols].where(df[bill_cols] > 0)
ratio = df[payamt_cols].values / bill_pos.values
with np.errstate(invalid="ignore"):
    sem_fatura = np.isnan(ratio).all(axis=1)
    avg_ratio = np.full(len(df), np.nan)
    avg_ratio[~sem_fatura] = np.nanmean(ratio[~sem_fatura], axis=1)

df["SEM_FATURA_POSITIVA"] = sem_fatura.astype(int)
log["sem_fatura_positiva"] = int(sem_fatura.sum())
print(f"Clientes sem nenhuma fatura positiva: {log['sem_fatura_positiva']} "
      f"({log['sem_fatura_positiva']/len(df)*100:.2f}%)")
print(pd.Series(avg_ratio).describe(percentiles=[.5, .95, .99]).round(3).to_string())

Clientes sem nenhuma fatura positiva: 936 (3.12%)
count    29029.000
mean         1.747
std         32.546
min          0.000
50%          0.085
95%          4.637
99%         17.846
max       4444.133


A cauda direita é extrema (p95 ≈ 4,6; máximo acima de 4.000), efeito de meses em que a fatura foi de poucos NT$ e o pagamento, muito superior. Winsorização em 5: preserva a ordenação de quem paga acima do faturado sem deixar que alguns extremos dominem média e desvio-padrão no escalonamento.

In [11]:
log["avg_pay_ratio_acima_de_5"] = int((pd.Series(avg_ratio) > 5).sum())
df["AVG_PAY_RATIO"] = pd.Series(avg_ratio, index=df.index).fillna(0.0).clip(upper=5)
print(f"Registros winsorizados: {log['avg_pay_ratio_acima_de_5']} "
      f"({log['avg_pay_ratio_acima_de_5']/len(df)*100:.2f}%)")

Registros winsorizados: 1349 (4.50%)


**2.3 `N_MESES_ATRASO`** — recorrência de atraso na janela de seis meses, sintetizando X6–X11 em um único indicador de tendência.

In [12]:
df["N_MESES_ATRASO"] = (df[pay_cols] > 0).sum(axis=1)
df.groupby("N_MESES_ATRASO")[TARGET].agg(["count", "mean"]).round(4)

,count,mean
N_MESES_ATRASO,,
0,19918,0.1171
1,4405,0.2988
2,1899,0.3876
3,1154,0.5087
4,951,0.5731
5,298,0.5738
6,1340,0.7030


## 3. Redução

`ID` já removido. As variáveis `BILL_AMT1..6` têm |r| < 0,02 com o alvo e ficam registradas como candidatas à remoção em uma segunda rodada de seleção (RFE ou importância de atributos) na fase de modelagem — são mantidas aqui porque sustentam `AVG_UTIL_RATIO` e porque ausência de correlação linear não implica ausência de relação não linear.

**Nenhum balanceamento (SMOTE) é aplicado nesta fase.** Reamostrar antes do split faria instâncias sintéticas derivadas de registros de teste contaminarem o treino.

## 4. Codificação de variáveis categóricas

`SEX`, `EDUCATION` e `MARRIAGE` são nominais → *one-hot* com `drop_first=True`. Sem a supressão da categoria de referência, a soma das dummies de um mesmo atributo é constante e igual a 1 — colinearidade perfeita, que inviabiliza a estimação dos coeficientes na Regressão Logística.

`PAY_1..PAY_6` permanecem numéricas: os códigos (−2 a 8) têm ordem intrínseca (grau de atraso).

In [13]:
df = pd.get_dummies(df, columns=["SEX", "EDUCATION", "MARRIAGE"],
                    prefix=["SEX", "EDU", "MAR"], drop_first=True)
print("Colunas após encoding:", df.shape[1])
print([c for c in df.columns if c.startswith(("SEX_", "EDU_", "MAR_"))])

Colunas após encoding: 31
['SEX_2', 'EDU_2', 'EDU_3', 'EDU_4', 'MAR_2', 'MAR_3']


## 5. Separação treino/teste

Divisão 80/20 **estratificada** pelo alvo, com semente fixa. Feita antes do escalonamento.

In [14]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
log["n_treino"], log["n_teste"] = len(X_train), len(X_test)
log["taxa_default_treino"] = round(float(y_train.mean()), 4)
log["taxa_default_teste"] = round(float(y_test.mean()), 4)
print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")
print(f"Inadimplentes -> treino: {y_train.mean():.4f} | teste: {y_test.mean():.4f}")

Treino: 23972 | Teste: 5993
Inadimplentes -> treino: 0.2213 | teste: 0.2213


## 6. Escalonamento

`StandardScaler` com `fit` **apenas no treino** e `transform` no teste — o contrário vazaria a média e o desvio-padrão do conjunto de teste para o treinamento. As dummies e a flag binária ficam fora, preservando a leitura 0/1.

In [15]:
num_cols = (["LIMIT_BAL", "AGE"] + pay_cols + bill_cols + payamt_cols +
            ["AVG_UTIL_RATIO", "AVG_PAY_RATIO", "N_MESES_ATRASO"])
scaler = StandardScaler()
X_train_scaled, X_test_scaled = X_train.copy(), X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])
log["n_colunas_finais"] = int(X_train_scaled.shape[1])
print(f"Variáveis preditoras finais: {X_train_scaled.shape[1]}")
X_train_scaled.head()

Variáveis preditoras finais: 30


,LIMIT_BAL,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,AVG_UTIL_RATIO,SEM_FATURA_POSITIVA,AVG_PAY_RATIO,N_MESES_ATRASO,SEX_2,EDU_2,EDU_3,EDU_4,MAR_2,MAR_3
27420,1.022024,-0.050264,-1.771947,-1.566494,-1.544266,-1.530037,-1.538794,-1.492534,-0.700030,-0.694464,-0.678531,-0.674057,-0.665085,-0.652938,-0.338846,-0.245685,-0.294973,-0.320892,-0.318684,-0.289319,-1.062663,1,-0.555385,-0.536972,False,True,False,False,False,False
15350,-0.903020,0.167213,1.793738,1.781057,1.818359,1.906811,2.010259,2.005012,-0.049079,-0.005582,0.032115,0.072324,-0.220084,-0.208968,-0.200960,-0.183578,-0.294973,-0.093938,-0.318684,-0.223376,1.275829,0,-0.529670,3.330712,True,False,True,False,False,True
26641,-0.672015,-1.355126,-0.880526,-0.729606,-0.703609,-0.670825,0.235732,0.256239,-0.693968,-0.688208,-0.672134,-0.660258,-0.650460,-0.645486,-0.312228,-0.227301,-0.244835,-0.291254,-0.318684,-0.232278,-1.041679,0,0.346009,-0.536972,False,False,False,False,True,False
21618,-0.903020,-1.028910,-0.880526,-0.729606,-0.703609,-0.670825,-0.651531,-0.618147,-0.694705,-0.665187,-0.672912,-0.667997,-0.665085,-0.639846,-0.214269,-0.229537,-0.272953,-0.320892,-0.266810,-0.289319,-1.024589,0,0.475358,-0.536972,True,True,False,False,True,False
11570,0.252006,-0.593956,-1.771947,-1.566494,-1.544266,-1.530037,-1.538794,-1.492534,-0.037938,0.009043,-0.321904,1.244053,1.513376,1.516122,0.877787,0.784923,6.772372,0.868346,1.428331,-0.050330,0.138043,0,0.281748,-0.536972,False,True,False,False,True,False


## 7. Exportação e verificação

In [16]:
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports", exist_ok=True)
X_train_scaled.assign(**{TARGET: y_train.values}).to_csv(
    "data/processed/treino_processado.csv", index=False)
X_test_scaled.assign(**{TARGET: y_test.values}).to_csv(
    "data/processed/teste_processado.csv", index=False)
with open("reports/estatisticas_preprocessamento.json", "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2, ensure_ascii=False)
print("Arquivos gravados em data/processed/")
print(json.dumps(log, indent=2, ensure_ascii=False))

Arquivos gravados em data/processed/
{
  "linhas_brutas": 30000,
  "colunas_brutas": 25,
  "valores_ausentes": 0,
  "duplicatas_removidas": 35,
  "linhas_apos_limpeza": 29965,
  "education_nao_documentado": 345,
  "marriage_nao_documentado": 54,
  "outliers_iqr_LIMIT_BAL": 167,
  "outliers_iqr_AGE": 272,
  "sem_fatura_positiva": 936,
  "avg_pay_ratio_acima_de_5": 1349,
  "n_treino": 23972,
  "n_teste": 5993,
  "taxa_default_treino": 0.2213,
  "taxa_default_teste": 0.2213,
  "n_colunas_finais": 30
}


In [17]:
# Verificações de sanidade do resultado
assert X_train_scaled.shape[1] == X_test_scaled.shape[1]
assert list(X_train_scaled.columns) == list(X_test_scaled.columns)
assert not X_train_scaled.isna().any().any()
assert not X_test_scaled.isna().any().any()
assert abs(y_train.mean() - y_test.mean()) < 0.01
media = X_train_scaled[num_cols].mean().abs().max()
desvio = (X_train_scaled[num_cols].std() - 1).abs().max()
print(f"Treino padronizado -> |média| máx: {media:.2e} | |desvio-1| máx: {desvio:.2e}")
print("Todas as verificações passaram.")

Treino padronizado -> |média| máx: 7.71e-17 | |desvio-1| máx: 2.09e-05
Todas as verificações passaram.


---

O dicionário de dados das 31 colunas resultantes está em `docs/dicionario_de_dados.md` (Tabela 6 do artigo).